# Splatter Image - Google Colab Backend Server
Run the cells below to start a FastAPI server on a free Google Colab GPU.
After running the last cell, copy the Cloudflare URL into your DJ application.

In [ ]:
!git clone https://github.com/szymanowiczs/splatter-image.git
!pip install -q rembg fastapi uvicorn python-multipart plyfile
!pip install -q einops imageio imageio-ffmpeg omegaconf
!wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
import torch
from huggingface_hub import hf_hub_download
print("Downloading Splatter Image AI model (this will take a minute or two)...")
model_path = hf_hub_download(repo_id="szymanowiczs/splatter-image-multi-category-v1", filename="model_latest.pth")
print("Model downloaded successfully to:", model_path)

In [ ]:
%%writefile server.py
import os
import io
import uuid
import sys
import numpy as np
from PIL import Image
import torch
from omegaconf import OmegaConf
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import rembg

SPLATTER_DIR = "/content/splatter-image"
sys.path.append(SPLATTER_DIR)

from scene.gaussian_predictor import GaussianSplatPredictor
from utils.app_utils import remove_background, resize_foreground, set_white_background, resize_to_128, to_tensor, export_to_obj
from utils.camera_utils import get_loop_cameras

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_cfg = OmegaConf.load(os.path.join(SPLATTER_DIR, "gradio_config.yaml"))
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(repo_id="szymanowiczs/splatter-image-multi-category-v1", filename="model_latest.pth")

model = GaussianSplatPredictor(model_cfg)
ckpt_loaded = torch.load(model_path, map_location=device)
model.load_state_dict(ckpt_loaded["model_state_dict"])
model.to(device)
model.eval()

print("Initializing rembg... (downloading u2net weights if needed)")
rembg_session = rembg.new_session()
source_camera = torch.from_numpy(get_loop_cameras()[0]).transpose(0, 1).unsqueeze(0).to(device)
print("--- BACKEND READY ---")

def preprocess_image(input_image: Image.Image, preprocess_background=True, foreground_ratio=0.65):
    if preprocess_background:
        image = input_image.convert("RGB")
        image = remove_background(image, rembg_session)
        image = resize_foreground(image, foreground_ratio)
        image = set_white_background(image)
    else:
        image = input_image
        if image.mode == "RGBA":
            image = set_white_background(image)
    image = resize_to_128(image)
    return image

@torch.no_grad()
def reconstruct_and_export(image: np.ndarray):
    image_tensor = to_tensor(image).unsqueeze(0).to(device)
    reconstruction_unactivated = model(
        image_tensor,
        source_camera,
        source_cv2wT_quat=None,
        focals_pixels=None,
        activate_output=True
    )
    ply_out_path = f"/tmp/splat_{uuid.uuid4().hex}.ply"
    export_to_obj(reconstruction_unactivated, ply_out_path)
    return ply_out_path

@app.post("/process_image")
async def process_image(file: UploadFile = File(...), remove_bg: bool = True):
    print(f"Received image: {file.filename}, remove_bg={remove_bg}")
    image_data = await file.read()
    input_image = Image.open(io.BytesIO(image_data))
    preprocessed = preprocess_image(input_image, preprocess_background=remove_bg)
    ply_out_path = reconstruct_and_export(np.array(preprocessed))
    return FileResponse(ply_out_path, media_type="application/octet-stream")


In [ ]:
import time
import re
import subprocess
import sys
import socket

print("Starting Uvicorn server in the background...")
!nohup {sys.executable} -m uvicorn server:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &

print("Waiting for server to fully initialize and open port 8000 (this can take 30-60 seconds if downloading rembg weights)...")
while True:
    try:
        with socket.create_connection(('127.0.0.1', 8000), timeout=1):
            break
    except OSError:
        time.sleep(1)
print("Port 8000 is open!")

print("Starting Cloudflare tunnel...")
!nohup ./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000 > cloudflare.log 2>&1 &
time.sleep(5)

log = open("cloudflare.log").read()
url = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log)
if url:
    print("\n========================================================")
    print("✅ SUCCESS! Your Cloudflare URL is ready:")
    print(url.group())
    print("========================================================")
    print("Copy the above URL and paste it into the DJ App's Backend Settings.")
else:
    print("Failed to get Cloudflare URL. Check cloudflare.log")


In [ ]:
## If you get a 502 error, run this cell to print the backend error log!
!cat uvicorn.log